In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import qmc
from scipy.spatial import distance

### Function Description

You’re tasked with optimising an ML model by tuning six hyperparameters, for example learning rate, regularisation strength or number of hidden layers. The function you’re maximising is the model’s performance score (such as accuracy or F1), but since the relationship between inputs and output isn’t known, it’s treated as a black-box function. 

Because this is a commonly used model, you might benefit from researching best practices or literature to guide your initial search space. Your goal is to find the combination of hyperparameters that yields the highest possible performance.

In [2]:
# Load data
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy').ravel()

In [3]:
df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6']) 
df['target'] = Y

print(df.describe())

              x1         x2         x3         x4         x5         x6  \
count  30.000000  30.000000  30.000000  30.000000  30.000000  30.000000   
mean    0.510852   0.395803   0.389609   0.512819   0.467201   0.484704   
std     0.303946   0.252579   0.314041   0.311966   0.314733   0.269343   
min     0.057896   0.011813   0.003635   0.073659   0.014944   0.051100   
25%     0.203026   0.197771   0.075285   0.220787   0.190704   0.276764   
50%     0.542122   0.387701   0.318555   0.459767   0.457732   0.552544   
75%     0.783687   0.543137   0.662281   0.826525   0.719142   0.707072   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.951014   

          target  
count  30.000000  
mean    0.219607  
std     0.307294  
min     0.002701  
25%     0.017954  
50%     0.078631  
75%     0.274431  
max     1.364968  


In [4]:
# Kernel Setup (6 Dimensions)
# normalize_y=False because targets are positive scores
initial_length_scales = np.array([1.0] * 6)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=False, 
    n_restarts_optimizer=50,  # Increased for 6D stability
    random_state=42
)

model.fit(X, Y)

# 6D Monte Carlo Random Search Space
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 6))

mean, std = model.predict(search_grid, return_std=True)

# Acquisition Function (UCB Beta Sweep)
betas = [0.1, 0.25, 0.5, 0.75, 1.0 , 1.5, 2.0, 2.5]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 'x5': bp[4], 'x6': bp[5],
        'Pred Mean': mean[best_idx],
        'Uncert': std[best_idx]
    })

# 6. Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 7: 6D HYPERPARAMETER OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 7: 6D HYPERPARAMETER OPTIMIZATION ---
  Beta     x1     x2     x3     x4     x5     x6  Pred Mean  Uncert
0.1000 0.0175 0.2132 0.4250 0.0449 0.3277 0.8870     1.5995  0.2621
0.2500 0.0072 0.0805 0.7596 0.0335 0.3999 0.9858     1.5891  0.3231
0.5000 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334
0.7500 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334
1.0000 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334
1.5000 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334
2.0000 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334
2.5000 0.1020 0.0594 0.1343 0.0118 0.3348 0.9699     1.5860  0.3334

--- Optimized Kernel Parameters ---
Matern(length_scale=[2.82, 1.94, 10, 1.16, 0.531, 1.24], nu=2.5) + WhiteKernel(noise_level=0.00355)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 1 submission: x1 = 0.1020, x2 = 0.0594, x3 = 0.1343, x4 = 0.0118, x5 = 0.3348, x6 = 0.9699

* The Problem Framing: 6 hyperparameters for a machine learning model. The target is a positive performance score (e.g., F1/Accuracy) that we want to maximize.
    * The Golden Clue: The landscape is mostly flat/mediocre, but contains a massive "Unicorn" peak (Index 6 scored 1.365, while the rest averaged near 0.10).
    * The Danger: 6D space is incredibly vast. Random search alone will almost certainly miss the narrow optimal peak.

* The Surrogate Model: Matern 2.5 + 6D ARD + High Restarts.
    * normalize_y=False: Since the targets are positive scores, we disabled normalization. This forces the model to assume unexplored space yields a 0.0 score, preventing it from wildly guessing that empty corners hold magical 2.0 scores without proof.
    * High Restarts (50): Increased the optimizer restarts to ensure it found the true ARD length scales in the bumpy 6D log-marginal-likelihood space.

* Key Insights from the Model: The "Dead Knob" (x3​): The length scale for x3​ hit the absolute maximum bound (10.0), changing x3​ has zero impact on the model's performance score.

* The Highly Sensitive Variable (x5​): With a length scale of 0.531, x5​ is the most volatile hyperparameter (potentially Learning Rate). The model is extremely careful to keep it anchored near 0.33.

* The Chosen Query (Beta 0.5+): [0.1020, 0.0594, 0.1343, 0.0118, 0.3348, 0.9699]
    * It safely anchors the sensitive variables while aggressively pushing the boundaries on x6​ and x4​ to hunt for the absolute global maximum.


In [5]:
# ---------------------------------------------------
# WEEK1: NEW DATA HERE
# ---------------------------------------------------
w1_new_input = [0.102 , 0.0594, 0.1343, 0.0118, 0.3348, 0.9699]
w1_new_output = 0.5670062028719492

new_X = np.array(w1_new_input).reshape(1, -1)
new_Y = np.atleast_1d(w1_new_output)

X_updated_w1 = np.vstack((X, new_X))
Y_updated_w1 = np.append(Y, new_Y)

df = pd.DataFrame(X_updated_w1, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w1

print(df.describe())
print(df.head(31))

              x1         x2         x3         x4         x5         x6  \
count  31.000000  31.000000  31.000000  31.000000  31.000000  31.000000   
mean    0.497663   0.384951   0.381373   0.496657   0.462930   0.500355   
std     0.307727   0.255578   0.312149   0.319650   0.310355   0.278786   
min     0.057896   0.011813   0.003635   0.011800   0.014944   0.051100   
25%     0.185615   0.168987   0.084639   0.218099   0.190980   0.298887   
50%     0.541241   0.377440   0.295542   0.449982   0.420428   0.585236   
75%     0.777431   0.538107   0.644549   0.820098   0.718908   0.720049   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.969900   

          target  
count  31.000000  
mean    0.230813  
std     0.308505  
min     0.002701  
25%     0.018039  
50%     0.083747  
75%     0.375144  
max     1.364968  
          x1        x2        x3        x4        x5        x6    target
0   0.272624  0.324495  0.897109  0.832951  0.154063  0.795864  0.604433
1   0.5

In [6]:
initial_length_scales = np.array([1.0] * 6)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=False, 
    n_restarts_optimizer=50,  # Increased for 6D stability
    random_state=42
)

model.fit(X_updated_w1, Y_updated_w1)

np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 6))

mean, std = model.predict(search_grid, return_std=True)

# Acquisition Function (UCB Beta Sweep)
betas = [0.1, 0.25, 0.5, 0.75, 1.0 , 1.5, 2.0, 2.5, 3.0, 10.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 'x5': bp[4], 'x6': bp[5],
        'Pred Mean': mean[best_idx],
        'Uncert': std[best_idx]
    })

df_sweep = pd.DataFrame(sweep_results)

print("--- FUNCTION 7: 6D HYPERPARAMETER OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 7: 6D HYPERPARAMETER OPTIMIZATION ---
     Beta       x1       x2       x3       x4       x5       x6  Pred Mean   Uncert
 0.100000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 0.250000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 0.500000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 0.750000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 1.000000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 1.500000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 2.000000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 2.500000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
 3.000000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080
10.000000 0.013343 0.437831 0.161431 0.140260 0.001516 0.997991   0.674112 0.291080

--- Optimized Kernel Par

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 2 Submission:

* **The Observation**: Beta sweep across [0.1 → 10.0] produces identical suggestions, revealing a **highly peaked GP posterior** where one region dominates all others.

* **Why This Happens**: The top 10 candidate points cluster tightly in predicted mean (0.674 → 0.657), so adding `β × std` shifts the argmax negligibly. The model has converged on a narrow "sweet spot."

* **What It Means**: The GP is confident about where to look next—there's minimal exploration upside because uncertainty is concentrated locally. This is **not a bug**, but a sign the search is narrowing in on a promising region.

* **The Decision**: Accept the consensus point suggested across all betas: **[0.0133, 0.4378, 0.1614, 0.1403, 0.0015, 0.9980]** with predicted mean ≈ 0.674.

* **Next Week Plan**: Potentially switch to **local optimization** (scipy L-BFGS-B) instead of grid search to better explore UCB peaks and generate diverse suggestions per beta.

In [7]:
# ---------------------------------------------------
# WEEK2: NEW DATA HERE
# ---------------------------------------------------

w2_new_input = [0.013343, 0.437831, 0.161431, 0.140260, 0.001516, 0.997991]
w2_new_output = np.float64(0.12936979316188976)

w2_new_X = np.atleast_2d(w2_new_input)
w2_new_Y = np.atleast_1d(w2_new_output)

X_updated_w2 = np.vstack((X_updated_w1, w2_new_X))
Y_updated_w2 = np.append(Y_updated_w1, w2_new_Y)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_updated_w2_scaled = scaler_x.fit_transform(X_updated_w2)
Y_updated_w2_scaled = scaler_y.fit_transform(Y_updated_w2.reshape(-1, 1)).flatten()

df = pd.DataFrame(X_updated_w2, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w2

print(df.describe())
print(df.head(32))

              x1         x2         x3         x4         x5         x6  \
count  32.000000  32.000000  32.000000  32.000000  32.000000  32.000000   
mean    0.482528   0.386604   0.374500   0.485520   0.448510   0.515906   
std     0.314597   0.251596   0.309525   0.320701   0.316016   0.288016   
min     0.013343   0.011813   0.003635   0.011800   0.001516   0.051100   
25%     0.169145   0.182266   0.093993   0.206201   0.181337   0.321010   
50%     0.534869   0.387701   0.293110   0.435183   0.417030   0.602159   
75%     0.771175   0.533076   0.626816   0.813672   0.718674   0.728771   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.997991   

          target  
count  32.000000  
mean    0.227643  
std     0.304018  
min     0.002701  
25%     0.018124  
50%     0.088196  
75%     0.325018  
max     1.364968  
          x1        x2        x3        x4        x5        x6    target
0   0.272624  0.324495  0.897109  0.832951  0.154063  0.795864  0.604433
1   0.5

In [8]:
initial_length_scales = np.array([1.0] * 6)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=False, 
    n_restarts_optimizer=50,
    random_state=42
)

model.fit(X_updated_w2_scaled, Y_updated_w2_scaled)

def acquisition_function(x_trial, beta, model):
    x_trial = x_trial.reshape(1, -1)
    mean, std = model.predict(x_trial, return_std=True)
    ucb = mean[0] + (beta * std[0])
    return -float(ucb)

betas = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

# Generate 8,192 quasi-random points to scout the 6D volume evenly
sampler = qmc.Sobol(d=6, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# 5. FORMAT AND PRINT RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---
    Beta       x1       x2       x3       x4       x5       x6  Pred Mean  Uncertainty
0.100000 0.015662 0.445073 0.286680 0.224845 0.402837 0.731627   1.373862     0.037217
0.250000 0.002780 0.431693 0.298267 0.226377 0.399094 0.731267   1.372397     0.045584
0.500000 0.000000 0.402782 0.323532 0.228916 0.392340 0.730133   1.367930     0.057558
0.750000 0.000000 0.360014 0.361648 0.230636 0.385414 0.727987   1.360514     0.069425
1.000000 0.000000 0.618876 0.896264 0.230551 0.378656 0.724424   1.350023     0.081327
1.500000 0.000000 0.618885 0.896406 0.221085 0.366051 0.710900   1.320164     0.105177
2.000000 0.000000 0.618931 0.896481 0.203069 0.356732 0.693015   1.277798     0.129410
3.000000 0.000000 0.223828 0.717266 0.165630 0.344617 0.660659   1.175331     0.170803

--- Optimized Kernel Parameters ---
Matern(length_scale=[3.49, 100, 100, 1.08, 0.599, 0.783], nu=2.5) + WhiteKernel(noise_level=1e-05)


### Week 3 submission:

* Current Progress: Didn't improve the recipe compared to last time - the output was significantly worse. 

* Had to recreate the whole model from zero because the output was completely non-sensical. Probably the last query was a complete waste. 

* Changing β=0.1 to try to verify if we can find a point that's close to the current max. 

* Next Query Coordinate (β=0.1): [0.015662 , 0.445073 , 0.286680 , 0.224845 , 0.402837 , 0.731627]

In [9]:
# ---------------------------------------------------
# WEEK3: NEW DATA HERE
# ---------------------------------------------------

w3_new_input = 	[0.015662, 0.445073, 0.286680, 0.224845, 0.402837, 0.731627]
w3_new_output = 1.5898428849757327

w3_new_X = np.atleast_2d(w3_new_input)
w3_new_Y = np.atleast_1d(w3_new_output)

X_updated_w3 = np.vstack((X_updated_w2, w3_new_X))
Y_updated_w3 = np.append(Y_updated_w2, w3_new_Y)

X_updated_w3_scaled = scaler_x.fit_transform(X_updated_w3)
Y_updated_w3_scaled = scaler_y.fit_transform(Y_updated_w3.reshape(-1, 1))

df = pd.DataFrame(X_updated_w3, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w3

print(df.describe())

              x1         x2         x3         x4         x5         x6  \
count  33.000000  33.000000  33.000000  33.000000  33.000000  33.000000   
mean    0.468380   0.388375   0.371839   0.477621   0.447126   0.522443   
std     0.320130   0.247842   0.305033   0.318896   0.311141   0.285956   
min     0.013343   0.011813   0.003635   0.011800   0.001516   0.051100   
25%     0.148647   0.195545   0.103348   0.218079   0.190428   0.343133   
50%     0.528497   0.397962   0.290678   0.420385   0.413631   0.619082   
75%     0.764919   0.528045   0.609084   0.807245   0.718440   0.730660   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.997991   

          target  
count  33.000000  
mean    0.268922  
std     0.381796  
min     0.002701  
25%     0.018209  
50%     0.092645  
75%     0.475396  
max     1.589843  


In [10]:
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-10, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=False, 
    n_restarts_optimizer=50,
    random_state=42
)

model.fit(X_updated_w3_scaled, Y_updated_w3_scaled)

betas = [0.05, 0.10, 0.25, 0.5, 0.75, 1.0]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# 5. FORMAT AND PRINT RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)


--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---
    Beta        x1       x2       x3       x4       x5       x6  Pred Mean  Uncertainty
0.050000 -0.000000 0.617396 0.886429 0.233272 0.381359 0.720564   1.478090     0.109513
0.100000 -0.000000 0.617396 0.886429 0.233402 0.380654 0.720096   1.478040     0.110170
0.250000 -0.000000 0.617396 0.886429 0.233710 0.378398 0.718507   1.477651     0.112369
0.500000 -0.000000 0.617396 0.886429 0.233759 0.374161 0.715108   1.475924     0.116908
0.750000 -0.000000 0.617396 0.886429 0.232753 0.369375 0.710477   1.472233     0.122766
1.000000 -0.000000 0.617396 0.886429 0.229996 0.364201 0.704285   1.465692     0.130200

--- Optimized Kernel Parameters ---
Matern(length_scale=[3.7, 7.8e+03, 1e+04, 1.33, 0.763, 0.996], nu=2.5) + WhiteKernel(noise_level=0.0422)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 4 submission:

Next point to query: [0.000000 , 0.617396 , 0.886429 , 0.233402 , 0.380654 , 0.720096] beta = 0.1 (BO)

* Found a new maximum, keep optimising - initially using the same beta = 0.1
* Implemented a parallel method using Neural Networks from different tests run, I don't have the same confidence as per BOs. 

In [11]:
# ---------------------------------------------------
# WEEK4: NEW DATA HERE
# ---------------------------------------------------

w4_new_input = 	[0.000000, 0.617396, 0.886429, 0.233402, 0.380654, 0.720096]
w4_new_output = 0.7877764225087719

w4_new_X = np.atleast_2d(w4_new_input)
w4_new_Y = np.atleast_1d(w4_new_output)

X_updated_w4 = np.vstack((X_updated_w3, w4_new_X))
Y_updated_w4 = np.append(Y_updated_w3, w4_new_Y)

X_updated_w4_scaled = scaler_x.fit_transform(X_updated_w4)
Y_updated_w4_scaled = scaler_y.fit_transform(Y_updated_w4.reshape(-1, 1))

df = pd.DataFrame(X_updated_w4, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w4

print(df.describe())

              x1         x2         x3         x4         x5         x6  \
count  34.000000  34.000000  34.000000  34.000000  34.000000  34.000000   
mean    0.454605   0.395111   0.386974   0.470438   0.445171   0.528257   
std     0.325315   0.247199   0.313072   0.316808   0.306602   0.283623   
min     0.000000   0.011813   0.003635   0.011800   0.001516   0.051100   
25%     0.126312   0.197771   0.110694   0.218089   0.190704   0.344457   
50%     0.502660   0.417896   0.293110   0.391215   0.408234   0.626393   
75%     0.755261   0.543137   0.662281   0.794507   0.711722   0.730030   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.997991   

          target  
count  34.000000  
mean    0.284182  
std     0.386354  
min     0.002701  
25%     0.018992  
50%     0.094028  
75%     0.506192  
max     1.589843  


In [12]:
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-10, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=False, 
    n_restarts_optimizer=50,
    random_state=42
)

model.fit(X_updated_w4_scaled, Y_updated_w4_scaled)

betas = [0.05, 0.10, 0.25, 0.5, 0.75, 1.0]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# 5. FORMAT AND PRINT RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)


--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---
    Beta       x1       x2       x3       x4       x5       x6  Pred Mean  Uncertainty
0.050000 0.000000 0.386274 0.056879 0.186061 0.386327 0.742046   1.677666     0.077379
0.100000 0.000000 0.385267 0.056879 0.184393 0.386189 0.742278   1.677562     0.078762
0.250000 0.000000 0.382211 0.056879 0.178915 0.385826 0.742956   1.676799     0.083105
0.500000 0.000000 0.377072 0.056879 0.168024 0.385415 0.743934   1.673826     0.090987
0.750000 0.000000 0.372032 0.056879 0.154584 0.385271 0.744484   1.668375     0.099682
1.000000 0.000000 0.367294 0.056879 0.138230 0.385373 0.744230   1.660012     0.109219

--- Optimized Kernel Parameters ---
Matern(length_scale=[3.23, 1.14, 9.97e+03, 2.09, 0.924, 1.24], nu=2.5) + WhiteKernel(noise_level=0.00301)


### Week 5 submission

Next point to query: [0.000000 0.386274 0.056879 0.186061 0.386327 0.742046] beta = 0.05

* Huge decrease in performance, very surprising 
* If persists, needs to be troubleshooted

In [13]:
# ---------------------------------------------------
# WEEK5: NEW DATA HERE
# ---------------------------------------------------

w5_new_input = 	[0.000000, 0.386274, 0.056879, 0.186061, 0.386327, 0.742046]
w5_new_output = 1.3773445470113153

w5_new_X = np.atleast_2d(w5_new_input)
w5_new_Y = np.atleast_1d(w5_new_output)

X_updated_w5 = np.vstack((X_updated_w4, w5_new_X))
Y_updated_w5 = np.append(Y_updated_w4, w5_new_Y)

X_updated_w5_scaled = scaler_x.fit_transform(X_updated_w5)
Y_updated_w5_scaled = scaler_y.fit_transform(Y_updated_w5.reshape(-1, 1))

df = pd.DataFrame(X_updated_w5, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w5

print(df.head(36))
# print(df.describe())

          x1        x2        x3        x4        x5        x6    target
0   0.272624  0.324495  0.897109  0.832951  0.154063  0.795864  0.604433
1   0.543003  0.924694  0.341567  0.646486  0.718440  0.343133  0.562753
2   0.090832  0.661529  0.065931  0.258577  0.963453  0.640265  0.007503
3   0.118867  0.615055  0.905816  0.855300  0.413631  0.585236  0.061424
4   0.630218  0.838097  0.680013  0.731895  0.526737  0.348429  0.273047
5   0.764919  0.255883  0.609084  0.218079  0.322943  0.095794  0.083747
6   0.057896  0.491672  0.247422  0.218118  0.420428  0.730970  1.364968
7   0.195252  0.079227  0.554580  0.170567  0.014944  0.107032  0.092645
8   0.642303  0.836875  0.021793  0.101488  0.683071  0.692416  0.017870
9   0.789943  0.195545  0.575623  0.073659  0.259049  0.051100  0.033565
10  0.528497  0.457424  0.360096  0.362046  0.816891  0.637476  0.073516
11  0.722615  0.011813  0.063646  0.165173  0.079244  0.359952  0.206310
12  0.075665  0.334502  0.132733  0.608312  0.91838

In [14]:
model.fit(X_updated_w5_scaled, Y_updated_w5_scaled)

betas = [0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    bounds = [scaler_x.transform([[0,0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Phase C: Reverse the scaling to return physical hyperparameter suggestions
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'x6': best_x_raw[5],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

# 5. FORMAT AND PRINT RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- 6D HYPERPARAMETER OPTIMIZATION SWEEP ---
    Beta        x1       x2       x3       x4       x5       x6  Pred Mean  Uncertainty
0.500000 -0.000000 0.426004 0.334553 0.215091 0.411330 0.776453   1.470631     0.104380
0.750000 -0.000000 0.425106 0.334553 0.217686 0.414384 0.791733   1.465183     0.112938
1.000000 -0.000000 0.424093 0.334553 0.220719 0.418000 0.811709   1.453474     0.126217
1.500000 -0.000000 0.422100 0.334553 0.225740 0.425533 0.857444   1.409793     0.160960
2.000000 -0.000000 0.420529 0.334553 0.228089 0.432934 0.904455   1.345693     0.197615
2.500000 -0.000000 0.419657 0.895824 0.227906 0.440300 0.951306   1.268598     0.231939
3.000000 -0.000000 0.419657 0.895824 0.225555 0.447703 0.997954   1.184103     0.262721

--- Optimized Kernel Parameters ---
Matern(length_scale=[4.54, 1.21, 5.22e+03, 1.95, 1.02, 1.68], nu=2.5) + WhiteKernel(noise_level=0.0352)


### Week 6 submission

Next point to query [0.422100 0.334553 0.225740 0.425533 0.857444] beta = 1.50 

* Last round with high exploitation was not succesful, so we are shifting into exploration
* The beta was chosen based on the highest upper bound (pred mean + uncertainty)


In [15]:
# ---------------------------------------------------
# WEEK6: NEW DATA HERE
# ---------------------------------------------------

w6_new_input = 	[0.000000, 0.422100, 0.334553, 0.225740, 0.425533, 0.857444]
w6_new_output = 1.1986521346801935

w6_new_X = np.atleast_2d(w6_new_input)
w6_new_Y = np.atleast_1d(w6_new_output)

X_updated_w6 = np.vstack((X_updated_w5, w6_new_X))
Y_updated_w6 = np.append(Y_updated_w5, w6_new_Y)

X_updated_w6_scaled = scaler_x.fit_transform(X_updated_w6)
Y_updated_w6_scaled = scaler_y.fit_transform(Y_updated_w6.reshape(-1, 1))

df = pd.DataFrame(X_updated_w6, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w6

# print(df.describe())
print(df.head(36))

          x1        x2        x3        x4        x5        x6    target
0   0.272624  0.324495  0.897109  0.832951  0.154063  0.795864  0.604433
1   0.543003  0.924694  0.341567  0.646486  0.718440  0.343133  0.562753
2   0.090832  0.661529  0.065931  0.258577  0.963453  0.640265  0.007503
3   0.118867  0.615055  0.905816  0.855300  0.413631  0.585236  0.061424
4   0.630218  0.838097  0.680013  0.731895  0.526737  0.348429  0.273047
5   0.764919  0.255883  0.609084  0.218079  0.322943  0.095794  0.083747
6   0.057896  0.491672  0.247422  0.218118  0.420428  0.730970  1.364968
7   0.195252  0.079227  0.554580  0.170567  0.014944  0.107032  0.092645
8   0.642303  0.836875  0.021793  0.101488  0.683071  0.692416  0.017870
9   0.789943  0.195545  0.575623  0.073659  0.259049  0.051100  0.033565
10  0.528497  0.457424  0.360096  0.362046  0.816891  0.637476  0.073516
11  0.722615  0.011813  0.063646  0.165173  0.079244  0.359952  0.206310
12  0.075665  0.334502  0.132733  0.608312  0.91838

In [16]:
# Grab the 75th percentile value as our dynamic floor
threshold = np.percentile(Y_updated_w6, 75)
mask = Y_updated_w6 > threshold

X_exploit_scaled = X_updated_w6_scaled[mask]
Y_exploit_scaled = Y_updated_w6_scaled[mask]

active_indices = [2, 3, 4, 5]
X_train_4d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

# Train the focused 4D GP
initial_length_scales = np.array([1.0] * 4)

kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1e1))

model_4d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_4d.fit(X_train_4d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_4d = X_train_4d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_4d = []
for idx in active_indices:
    bounds_4d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_4d,
    args=(model_4d,),
    method='L-BFGS-B',
    bounds=bounds_4d
)

best_x_4d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_6d_scaled = np.zeros(6)

scaled_zero_x1 = scaled_lower[0] 

final_6d_scaled[0] = scaled_zero_x1           # x1: Frozen at absolute physical floor (0.0)
final_6d_scaled[1] = best_full_row_scaled[1]  # x2: Frozen at best known (Ghost parameter)
final_6d_scaled[2] = best_x_4d[0]             # x3: Optimized
final_6d_scaled[3] = best_x_4d[1]             # x4: Optimized
final_6d_scaled[4] = best_x_4d[2]             # x5: Optimized
final_6d_scaled[5] = best_x_4d[3]             # x6: Optimized

final_6d_raw = scaler_x.inverse_transform(final_6d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

# 8. Output the final recommendation
print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
feature_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
for name, val in zip(feature_names, final_6d_raw):
    print(f"{name}: {val:.6f}")

print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
x1: 0.000000
x2: 0.445073
x3: 0.193085
x4: 0.224845
x5: 0.403300
x6: 0.731627

Predicted Target Mean: 1.607302


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 7 submission

Next point to query [0.000000 0.445073 0.193085 0.224845 0.403300 0.731627] 

* Applied dimensionality freezing (same as function 8) - went from 6D to 4D, dropped x1 and x2. 

In [17]:
# ---------------------------------------------------
# WEEK7: NEW DATA HERE
# ---------------------------------------------------

w7_new_input = 	[0.000000, 0.445073, 0.193085, 0.224845, 0.403300, 0.731627]
w7_new_output = 1.4347027647452144

w7_new_X = np.atleast_2d(w7_new_input)
w7_new_Y = np.atleast_1d(w7_new_output)

X_updated_w7 = np.vstack((X_updated_w6, w7_new_X))
Y_updated_w7 = np.append(Y_updated_w6, w7_new_Y)

X_updated_w7_scaled = scaler_x.fit_transform(X_updated_w7)
Y_updated_w7_scaled = scaler_y.fit_transform(Y_updated_w7.reshape(-1, 1))

df = pd.DataFrame(X_updated_w7, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w7

# print(df.describe())
print(df.head(37))

          x1        x2        x3        x4        x5        x6    target
0   0.272624  0.324495  0.897109  0.832951  0.154063  0.795864  0.604433
1   0.543003  0.924694  0.341567  0.646486  0.718440  0.343133  0.562753
2   0.090832  0.661529  0.065931  0.258577  0.963453  0.640265  0.007503
3   0.118867  0.615055  0.905816  0.855300  0.413631  0.585236  0.061424
4   0.630218  0.838097  0.680013  0.731895  0.526737  0.348429  0.273047
5   0.764919  0.255883  0.609084  0.218079  0.322943  0.095794  0.083747
6   0.057896  0.491672  0.247422  0.218118  0.420428  0.730970  1.364968
7   0.195252  0.079227  0.554580  0.170567  0.014944  0.107032  0.092645
8   0.642303  0.836875  0.021793  0.101488  0.683071  0.692416  0.017870
9   0.789943  0.195545  0.575623  0.073659  0.259049  0.051100  0.033565
10  0.528497  0.457424  0.360096  0.362046  0.816891  0.637476  0.073516
11  0.722615  0.011813  0.063646  0.165173  0.079244  0.359952  0.206310
12  0.075665  0.334502  0.132733  0.608312  0.91838

In [18]:
# Grab the 75th percentile value as our dynamic floor
threshold = np.percentile(Y_updated_w7, 75)
mask = Y_updated_w7 > threshold

X_exploit_scaled = X_updated_w7_scaled[mask]
Y_exploit_scaled = Y_updated_w7_scaled[mask]

active_indices = [2, 3, 4, 5]
X_train_4d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

# Train the focused 4D GP
initial_length_scales = np.array([1.0] * 4)

kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1e1))

model_4d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_4d.fit(X_train_4d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_4d = X_train_4d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_4d = []
for idx in active_indices:
    bounds_4d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_4d,
    args=(model_4d,),
    method='L-BFGS-B',
    bounds=bounds_4d
)

best_x_4d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_6d_scaled = np.zeros(6)

scaled_zero_x1 = scaled_lower[0] 

final_6d_scaled[0] = scaled_zero_x1           # x1: Frozen at absolute physical floor (0.0)
final_6d_scaled[1] = best_full_row_scaled[1]  # x2: Frozen at best known (Ghost parameter)
final_6d_scaled[2] = best_x_4d[0]             # x3: Optimized
final_6d_scaled[3] = best_x_4d[1]             # x4: Optimized
final_6d_scaled[4] = best_x_4d[2]             # x5: Optimized
final_6d_scaled[5] = best_x_4d[3]             # x6: Optimized

final_6d_raw = scaler_x.inverse_transform(final_6d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

# 8. Output the final recommendation
print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_6d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[-0.000000, 0.445073, 0.293684, 0.224845, 0.389329, 0.681969]

Predicted Target Mean: 1.548210


### Week 8 submission

Next point to query [0.000000, 0.445073, 0.293684, 0.224845, 0.389329, 0.681969]

* Function looks like is working well, but as with 8D the output is not accurate with the prediction 
* The target mean is lower than our current max
* Need to understand variance

In [19]:
# ---------------------------------------------------
# WEEK8: NEW DATA HERE
# ---------------------------------------------------

w8_new_input = 	[0.000000, 0.445073, 0.293684, 0.224845, 0.389329, 0.681969]
w8_new_output = 1.6850169035049964

w8_new_X = np.atleast_2d(w8_new_input)
w8_new_Y = np.atleast_1d(w8_new_output)

X_updated_w8 = np.vstack((X_updated_w7, w8_new_X))
Y_updated_w8 = np.append(Y_updated_w7, w8_new_Y)

X_updated_w8_scaled = scaler_x.fit_transform(X_updated_w8)
Y_updated_w8_scaled = scaler_y.fit_transform(Y_updated_w8.reshape(-1, 1))

df = pd.DataFrame(X_updated_w8, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w8

print(df.describe())
# print(df.head(48))

              x1         x2         x3         x4         x5         x6  \
count  38.000000  38.000000  38.000000  38.000000  38.000000  38.000000   
mean    0.406751   0.398218   0.369350   0.443589   0.440535   0.551942   
std     0.338200   0.233769   0.302274   0.309581   0.289924   0.277651   
min     0.000000   0.011813   0.003635   0.011800   0.001516   0.051100   
25%     0.079457   0.217308   0.110694   0.218089   0.208411   0.351310   
50%     0.368718   0.429966   0.292181   0.300538   0.403068   0.638871   
75%     0.725368   0.526058   0.600719   0.750193   0.689444   0.731463   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.997991   

          target  
count  38.000000  
mean    0.404155  
std     0.511918  
min     0.002701  
25%     0.024398  
50%     0.114938  
75%     0.595076  
max     1.685017  


In [20]:
# Grab the 70th percentile value as our dynamic floor
threshold = np.percentile(Y_updated_w8, 70)
mask = Y_updated_w8 > threshold

X_exploit_scaled = X_updated_w8_scaled[mask]
Y_exploit_scaled = Y_updated_w8_scaled[mask]

active_indices = [2, 3, 4, 5]
X_train_4d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

# Train the focused 4D GP
initial_length_scales = np.array([1.0] * 4)

kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1e1))

model_4d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_4d.fit(X_train_4d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_4d = X_train_4d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_4d = []
for idx in active_indices:
    bounds_4d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_4d,
    args=(model_4d,),
    method='L-BFGS-B',
    bounds=bounds_4d
)

best_x_4d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_6d_scaled = np.zeros(6)

scaled_zero_x1 = scaled_lower[0] 

final_6d_scaled[0] = scaled_zero_x1           # x1: Frozen at absolute physical floor (0.0)
final_6d_scaled[1] = best_full_row_scaled[1]  # x2: Frozen at best known (Ghost parameter)
final_6d_scaled[2] = best_x_4d[0]             # x3: Optimized
final_6d_scaled[3] = best_x_4d[1]             # x4: Optimized
final_6d_scaled[4] = best_x_4d[2]             # x5: Optimized
final_6d_scaled[5] = best_x_4d[3]             # x6: Optimized

final_6d_raw = scaler_x.inverse_transform(final_6d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

# 8. Output the final recommendation
print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_6d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.000000, 0.445073, 0.347065, 0.210104, 0.384511, 0.673715]

Predicted Target Mean: 1.726122


### Week 9 submission

Next point to query [0.000000, 0.445073, 0.347065, 0.210104, 0.384511, 0.673715]

* New max point
* Model is working

In [21]:
# ---------------------------------------------------
# WEEK9: NEW DATA HERE
# ---------------------------------------------------

w9_new_input = [0.000000, 0.445073, 0.347065, 0.210104, 0.384511, 0.673715]
w9_new_output = 1.7549233300698086

w9_new_X = np.atleast_2d(w9_new_input)
w9_new_Y = np.atleast_1d(w9_new_output)

X_updated_w9 = np.vstack((X_updated_w8, w9_new_X))
Y_updated_w9 = np.append(Y_updated_w8, w9_new_Y)

X_updated_w9_scaled = scaler_x.fit_transform(X_updated_w9)
Y_updated_w9_scaled = scaler_y.fit_transform(Y_updated_w9.reshape(-1, 1))

df = pd.DataFrame(X_updated_w9, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6'])
df['target'] = Y_updated_w9

print(df.describe())
# print(df.head(39))

              x1         x2         x3         x4         x5         x6  \
count  39.000000  39.000000  39.000000  39.000000  39.000000  39.000000   
mean    0.396322   0.399420   0.368779   0.437602   0.439098   0.555065   
std     0.340017   0.230794   0.298291   0.307760   0.286224   0.274666   
min     0.000000   0.011813   0.003635   0.011800   0.001516   0.051100   
25%     0.071138   0.230167   0.118040   0.214092   0.225290   0.354190   
50%     0.319810   0.437831   0.293684   0.285009   0.402837   0.640265   
75%     0.724450   0.524071   0.592354   0.744094   0.687320   0.731298   
max     0.942451   0.924694   0.924571   0.961017   0.998655   0.997991   

          target  
count  39.000000  
mean    0.438791  
std     0.549498  
min     0.002701  
25%     0.027454  
50%     0.129370  
75%     0.607979  
max     1.754923  


In [22]:
# Grab the 70th percentile value as our dynamic floor
threshold = np.percentile(Y_updated_w8, 70)
mask = Y_updated_w8 > threshold

X_exploit_scaled = X_updated_w8_scaled[mask]
Y_exploit_scaled = Y_updated_w8_scaled[mask]

active_indices = [2, 3, 4, 5]
X_train_4d = X_exploit_scaled[:, active_indices]
y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 4)

kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1e1))

model_4d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_4d.fit(X_train_4d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_4d = X_train_4d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_4d = []
for idx in active_indices:
    bounds_4d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_4d,
    args=(model_4d,),
    method='L-BFGS-B',
    bounds=bounds_4d
)

best_x_4d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_6d_scaled = np.zeros(6)

scaled_zero_x1 = scaled_lower[0] 

final_6d_scaled[0] = scaled_zero_x1           # x1: Frozen at absolute physical floor (0.0)
final_6d_scaled[1] = best_full_row_scaled[1]  # x2: Frozen at best known (Ghost parameter)
final_6d_scaled[2] = best_x_4d[0]             # x3: Optimized
final_6d_scaled[3] = best_x_4d[1]             # x4: Optimized
final_6d_scaled[4] = best_x_4d[2]             # x5: Optimized
final_6d_scaled[5] = best_x_4d[3]             # x6: Optimized

final_6d_raw = scaler_x.inverse_transform(final_6d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_6d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.000000, 0.445694, 0.346779, 0.205411, 0.383770, 0.675571]

Predicted Target Mean: 1.858293


### Week 10 submission

Next point to query [0.000000, 0.445694, 0.346779, 0.205411, 0.383770, 0.675571]

* New max point